<a href="https://colab.research.google.com/github/ProjetosNanos/Cap3-PydanticAI/blob/main/PydanticAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1- Introducao ao Pydantic AI




In [1]:
pip install pydantic-ai

In [2]:
from pydantic import BaseModel, Field
from decimal import Decimal
from typing import Optional

class Produto(BaseModel):
    nome: str = Field(min_length=1, max_length=100)
    preco: Decimal = Field(gt=0, le=10000)
    descricao: Optional[str] = Field(None, max_length=500)
    categoria: str = Field(..., pattern=r'^[A-Za-z\s]+$')
    em_estoque: int = Field(ge=0)
    disponivel: bool = True


In [3]:
try:
    produto_invalido = Produto(nome="", preco=-5, categoria="1234", em_estoque=-10)
except Exception as e:
    print("Erros de validação:", e)

Erros de validação: 4 validation errors for Produto
nome
  String should have at least 1 character [type=string_too_short, input_value='', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/string_too_short
preco
  Input should be greater than 0 [type=greater_than, input_value=-5, input_type=int]
    For further information visit https://errors.pydantic.dev/2.12/v/greater_than
categoria
  String should match pattern '^[A-Za-z\s]+$' [type=string_pattern_mismatch, input_value='1234', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/string_pattern_mismatch
em_estoque
  Input should be greater than or equal to 0 [type=greater_than_equal, input_value=-10, input_type=int]
    For further information visit https://errors.pydantic.dev/2.12/v/greater_than_equal


In [4]:
produto_valido = Produto(nome="Fone de Ouvido", preco="199.99", categoria="Eletronicos", em_estoque=15)
print(produto_valido)

nome='Fone de Ouvido' preco=Decimal('199.99') descricao=None categoria='Eletronicos' em_estoque=15 disponivel=True


2- Pydantic AI e LLM


In [7]:
# 1. Instala dependências (zstd) e o Ollama
!sudo apt-get update && sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

# 2. Inicia servidor Ollama server em background e aguarda inicialização
import subprocess
import time

# Inicia o servidor e redireciona a saída
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Aguarda 10 segundos para o servidor subir
print("Aguardando o servidor Ollama iniciar...")
time.sleep(10)

# 3. Faz download do modelo
!ollama pull llama3

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 3,917 B in 1s (4,277 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3bu

In [ ]:
from pydantic import BaseModel
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIModel
from pydantic_ai.providers.ollama import OllamaProvider
import nest_asyncio

# Apply the patch to allow nested asyncio event loops
nest_asyncio.apply()

class CityLocation(BaseModel):
    city: str
    country: str

ollama_model = OpenAIModel(
    model_name='gpt-oss:20b',   # must match the model you pulled in Ollama
    provider=OllamaProvider(base_url='http://localhost:11434/v1'),
)

agent = Agent(model=ollama_model, output_type=CityLocation)

result = agent.run_sync('Where were the Olympics held in 2012?')
print(result.output)  # -> city='London' country='United Kingdom'

3- Ferramentas

In [ ]:
import random
from pydantic_ai import Agent, RunContext

agent = Agent(
    model=ollama_model,
    deps_type=str,
    system_prompt="Você é um jogo de dados: role o dado e veja se acerta o palpite."
)

@agent.tool_plain
def roll_dice() -> str:
    return str(random.randint(1, 6))

@agent.tool
def get_player_name(ctx: RunContext[str]) -> str:
    return ctx.deps

dice_result = agent.run_sync('Meu palpite é 4', deps='Carlos')
print(dice_result.output)

4- Pipeline de Agentes completos

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Protocol
from pydantic import BaseModel, Field
from pydantic_ai import Agent, RunContext
from pydantic_ai.models.openai import OpenAIModel
from pydantic_ai.providers.ollama import OllamaProvider

ollama_model = OpenAIModel(
    model_name='gpt-oss:20b',   # must match the model you pulled in Ollama
    provider=OllamaProvider(base_url='http://localhost:11434/v1'),
)


# --- Contrato para o seu banco (assíncrono) ---
class MedicalDatabase(Protocol):
    async def get_history(self, patient_id: int) -> str: ...
    async def get_patient_name(self, patient_id: int) -> str: ...


# --- Dependências do agente (injeção) ---
@dataclass
class RequestContext:
    patient_id: int
    db: MedicalDatabase


# --- Esquemas de entrada/saída ---
class AssistanceRequest(BaseModel):
    symptoms: str = Field(max_length=500)
    urgency: int = Field(ge=1, le=5)


class AssistanceResponse(BaseModel):
    advice: str
    risk_level: int = Field(ge=0, le=10)
    refer_to_specialist: bool


# --- Definição do agente ---
assist_agent = Agent(
    model=ollama_model,
    deps_type=RequestContext,
    output_type=AssistanceResponse,
    # Use "instructions" em vez de "system_prompt" para o comportamento atual recomendado
    instructions=(
        "Você é um assistente médico virtual. "
        "Avalie os sintomas de forma prudente, comunique incertezas, "
        "e forneça orientação geral e não-diagnóstica. "
        "Se houver sinais de emergência (ex.: dor torácica intensa, falta de ar importante), "
        "incentive procurar atendimento imediato (SAMU/192 ou emergência). "
        "Respeite o limite do escopo: não prescreva medicamentos."
    ),
)


# --- Tool que usa o contexto (RunContext[RequestContext]) ---
@assist_agent.tool
async def fetch_patient_history(ctx: RunContext[RequestContext]) -> str:
    """Busca histórico clínico do paciente no banco."""
    return await ctx.deps.db.get_history(patient_id=ctx.deps.patient_id)


# --- Prompt dinâmico: acrescenta contexto do paciente antes da execução ---
@assist_agent.system_prompt
async def add_patient_context(ctx: RunContext[RequestContext]) -> str:
    name = await ctx.deps.db.get_patient_name(ctx.deps.patient_id)
    return f"Contexto: paciente '{name}' (id={ctx.deps.patient_id})."


# --- Exemplo de execução (sync) ---
class FakeDB:
    async def get_history(self, patient_id: int) -> str:
        return "Hipertensão controlada; sem alergias registradas."
    async def get_patient_name(self, patient_id: int) -> str:
        return "José da Silva"

deps = RequestContext(patient_id=42, db=FakeDB())
question = AssistanceRequest(symptoms="Dor no peito e falta de ar ao esforço", urgency=5)
result = await assist_agent.run(
    f"Avalie: {question.model_dump_json()}",
    deps=deps,
)
print(result.output)

5- Multiplos agentes

In [ ]:
from pydantic_ai import Agent, RunContext
from pydantic_ai.usage import UsageLimits
import asyncio

joke_selection_agent = Agent(
    ollama_model,
    system_prompt='Use a `joke_factory` para gerar piadas, depois escolha a melhor e retorne apenas uma.'
)

joke_generation_agent = Agent(
    ollama_model,
    output_type=list[str]
)

@joke_selection_agent.tool
async def joke_factory(ctx: RunContext[None], count: int) -> list[str]:
    r = await joke_generation_agent.run(
        f'Por favor, gere {count} piadas.',
        usage=ctx.usage,
    )
    return r.output

async def main():
    result = await joke_selection_agent.run(
        'Conte uma piada.',
        usage_limits=UsageLimits(request_limit=5, total_tokens_limit=2000),
    )
    print(result.output)
    print(result.usage())

await main()

6- Teste de Agente
